# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Setup

In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# Token order:
# environment variable -> Colab Secret -> prompt as last resort
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# Exact file/directory structure from the working warehouse notebook.
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected to FlyRank warehouse.")
print("Development decision point: 2026-03-31")

Connected to FlyRank warehouse.
Development decision point: 2026-03-31


In [5]:
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [6]:
DECISION_DATE = "2026-03-31"

In [7]:
content_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        DATE_DIFF(
            'day',
            CAST(content_created_date AS DATE),
            DATE '{DECISION_DATE}'
        ) AS content_age_days

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

In [8]:
content_update_state = con.sql(
    f"""
    SELECT
        content_hash_id,

        CASE
            WHEN CAST(content_updated_date AS DATE)
                 <= DATE '{DECISION_DATE}'
            THEN DATE_DIFF(
                'day',
                CAST(content_updated_date AS DATE),
                DATE '{DECISION_DATE}'
            )
            ELSE NULL
        END AS days_since_last_update

    FROM {TABLES['dim_content']}

    WHERE CAST(content_created_date AS DATE)
          <= DATE '{DECISION_DATE}'
    """
).df()

In [9]:
content_state = content_state.merge(
    content_update_state,
    on="content_hash_id",
    how="left"
)

In [10]:
historical_features = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev30,

        SUM(gsc_clicks) AS clicks_prev30,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN
                SUM(gsc_sum_position)
                / SUM(gsc_impressions)
            ELSE NULL
        END AS avg_position_prev30

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-30'

      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update",
]

feature_frame = historical_features.merge(
    content_state,
    on="content_hash_id",
    how="left"
)

feature_frame = feature_frame[
    [
        "client_hash_id",
        "content_hash_id",
        *FEATURES
    ]
].copy()

feature_frame.head()

,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,786.0,1.0,5.922392,47,<NA>
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,5.941176,47,<NA>
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,324.0,0.0,5.129630,47,<NA>
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,763.0,1.0,4.826999,47,<NA>
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0,4.285714,47,<NA>


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will start with **Logistic Regression** because this lane has an observed binary outcome and the practical goal is to **rank pages for human review**. The model will produce a probability score for each page, which can be used to rank pages from higher to lower review priority.

This fits the five decision-time features in the approved feature frame: `imp_prev30`, `clicks_prev30`, `avg_position_prev30`, `content_age_days`, and `days_since_last_update`. Logistic Regression is also a simple and interpretable starting model, so its behavior can be inspected before considering a more complex model.

The model will be evaluated as a ranking system using **Precision@K**, using the same metric and evaluation setup as the Week-4 baseline. If a more complex model is considered later, it should earn its additional complexity by improving the comparison rather than being preferred automatically.

The model is intended to provide **observed, directional, decision-support information** about which pages may deserve review. It does not establish that updating a page will cause better performance.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### 2. Split design

I will use a **client-grouped train/test split** because multiple content pages can belong to the same client. Splitting individual rows could place pages from the same client in both the training and test sets, making the evaluation less honest because the model could learn client-specific patterns from the training data.

The split will therefore keep each `client_hash_id` entirely within either the training set or the test set. I will use an **80/20 split of clients** with a fixed random seed for reproducibility.

The held-out test set will be used for the Week-5 model evaluation and for evaluating the Week-4 baseline on the same pages. This keeps the model and baseline comparison on the same evaluation population.

This split is appropriate for the question because the goal is to determine whether the learned model can prioritize pages for human review without relying on information from the same client's pages appearing in both training and evaluation.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# ---------------------------------------------------------
# Task 2 — Client-grouped train/test split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        feature_frame,
        groups=feature_frame["client_hash_id"]
    )
)

train_df = feature_frame.iloc[train_idx].copy()
test_df = feature_frame.iloc[test_idx].copy()

# ---------------------------------------------------------
# Verify the split
# ---------------------------------------------------------

train_clients = set(train_df["client_hash_id"].dropna().unique())
test_clients = set(test_df["client_hash_id"].dropna().unique())

overlap_clients = train_clients.intersection(test_clients)

print(f"Total rows: {len(feature_frame):,}")
print(f"Training rows: {len(train_df):,}")
print(f"Test rows: {len(test_df):,}")

print(f"\nTraining clients: {len(train_clients):,}")
print(f"Test clients: {len(test_clients):,}")

print(f"\nClient overlap: {len(overlap_clients)}")

assert len(overlap_clients) == 0, "Client leakage detected!"

print("\nClient-grouped split verified: no client appears in both train and test.")

Total rows: 175,205
Training rows: 137,354
Test rows: 37,851

Training clients: 37
Test clients: 10

Client overlap: 0

Client-grouped split verified: no client appears in both train and test.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I will train the Logistic Regression model using the five approved decision-time features and the client-grouped training set from Task 2.

The outcome label represents whether a page's impressions declined in the following 30-day period:

`is_declining = imp_last30 < 0.8 × imp_prev30`

The model will produce a probability of decline for each test page. Pages will be ranked from highest to lowest predicted probability, because the practical goal is to prioritize pages for human review.

The Week-4 rule baseline will be evaluated on the **same held-out test pages** and with the **same Precision@K metric**. This provides a direct comparison between the transparent rule and the learned model.

The comparison will use Precision@20 and Precision@50, with the test-set base rate also reported. These metrics reflect the ranking use case: how many of the top-ranked pages are actually observed as declining.


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# Task 3 — Create the future outcome label
# ---------------------------------------------------------

future_performance = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_last30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-04-01'
      AND report_date <= DATE '2026-04-30'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Previous 30-day impressions are already in feature_frame.
# Add the future 30-day impressions.
model_df = feature_frame.merge(
    future_performance,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

# ---------------------------------------------------------
# Create the observed future outcome
# ---------------------------------------------------------

model_df["is_declining"] = (
    model_df["imp_last30"] < 0.8 * model_df["imp_prev30"]
)

# Pages without future search data cannot provide an observed
# outcome, so exclude them from model training/evaluation.
model_df = model_df[
    model_df["imp_last30"].notna()
].copy()

# ---------------------------------------------------------
# Recreate the same client-grouped split
# ---------------------------------------------------------

train_df = model_df[
    model_df["client_hash_id"].isin(train_clients)
].copy()

test_df = model_df[
    model_df["client_hash_id"].isin(test_clients)
].copy()

# ---------------------------------------------------------
# Prepare X and y
# ---------------------------------------------------------

X_train = train_df[FEATURES]
y_train = train_df["is_declining"].astype(int)

X_test = test_df[FEATURES]
y_test = test_df["is_declining"].astype(int)

# ---------------------------------------------------------
# Logistic Regression pipeline
# ---------------------------------------------------------

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

# ---------------------------------------------------------
# Model ranking score
# ---------------------------------------------------------

test_df["model_score"] = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Week-4 baseline score on the SAME test pages
# ---------------------------------------------------------

test_df["visibility_score"] = np.where(
    test_df["imp_prev30"] >= 500,
    1,
    0
)

test_df["stale_score"] = np.where(
    test_df["days_since_last_update"].notna()
    & (test_df["days_since_last_update"] >= 180),
    1,
    0
)

test_df["baseline_score"] = (
    test_df["visibility_score"] * 2
    + test_df["stale_score"]
)

# Match the Week-4 baseline tie-breaker:
# higher recent impressions first.
test_df["baseline_rank"] = (
    test_df
    .sort_values(
        ["baseline_score", "imp_prev30"],
        ascending=[False, False]
    )
    .groupby(lambda _: True)
    .cumcount()
    + 1
)

# ---------------------------------------------------------
# Precision@K
# ---------------------------------------------------------

def precision_at_k(df, score_col, k):
    ranked = df.sort_values(
        score_col,
        ascending=False
    ).head(k)

    return ranked["is_declining"].mean()

# ---------------------------------------------------------
# Compare baseline vs model
# ---------------------------------------------------------

results = []

for k in [20, 50]:
    results.append({
        "Method": "Week-4 Baseline",
        f"Precision@{k}": precision_at_k(
            test_df,
            "baseline_score",
            k
        )
    })

    results.append({
        "Method": "Logistic Regression",
        f"Precision@{k}": precision_at_k(
            test_df,
            "model_score",
            k
        )
    })

comparison = pd.DataFrame(results)

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training base rate:", y_train.mean())
print("Test base rate:", y_test.mean())

print("\nComparison:")
display(comparison)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 121203
Test rows: 36022
Training base rate: 0.4772819154641387
Test base rate: 0.41360835045250127

Comparison:


,Method,Precision@20,Precision@50
0,Week-4 Baseline,0.2,NaN
1,Logistic Regression,0.6,NaN
2,Week-4 Baseline,NaN,0.24
3,Logistic Regression,NaN,0.54


Both methods are evaluated on the same 36,022 test pages for which the future outcome is observable.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model's overall Precision@K does not show where its ranking decisions fail, so I will inspect individual errors and the features used by the model.

First, I will examine false positives and false negatives among the held-out test pages. This helps identify whether the model is systematically confusing certain types of pages.

Second, I will inspect the Logistic Regression coefficients. Because the features are standardized before fitting, the coefficient magnitudes provide a simple indication of which features the model relies on most strongly. Positive coefficients increase the predicted probability of decline, while negative coefficients decrease it.

This analysis is used to understand the model's behavior, not to claim that any feature causes a page to decline.


In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# --------------------------------------------------
# 1. Create predictions and classify errors
# --------------------------------------------------

test_df["predicted_decline"] = (
    test_df["model_score"] >= 0.50
).astype(int)

test_df["error_type"] = "Correct"

test_df.loc[
    (test_df["predicted_decline"] == 1) &
    (test_df["is_declining"] == 0),
    "error_type"
] = "False Positive"

test_df.loc[
    (test_df["predicted_decline"] == 0) &
    (test_df["is_declining"] == 1),
    "error_type"
] = "False Negative"


# --------------------------------------------------
# 2. Error counts
# --------------------------------------------------

error_counts = (
    test_df["error_type"]
    .value_counts()
    .reindex(
        ["Correct", "False Positive", "False Negative"],
        fill_value=0
    )
)

print("Error summary:")
display(error_counts.to_frame("count"))


# --------------------------------------------------
# 3. Show concrete false-positive cases
# --------------------------------------------------

false_positives = (
    test_df[
        test_df["error_type"] == "False Positive"
    ]
    .sort_values("model_score", ascending=False)
    [
        [
            "content_hash_id",
            "model_score",
            "is_declining",
            "imp_prev30",
            "clicks_prev30",
            "avg_position_prev30",
            "content_age_days",
            "days_since_last_update"
        ]
    ]
    .head(3)
)

print("Top 3 false positives:")
display(false_positives)


# --------------------------------------------------
# 4. Show concrete false-negative cases
# --------------------------------------------------

false_negatives = (
    test_df[
        test_df["error_type"] == "False Negative"
    ]
    .sort_values("model_score", ascending=True)
    [
        [
            "content_hash_id",
            "model_score",
            "is_declining",
            "imp_prev30",
            "clicks_prev30",
            "avg_position_prev30",
            "content_age_days",
            "days_since_last_update"
        ]
    ]
    .head(3)
)

print("Top 3 false negatives:")
display(false_negatives)


# --------------------------------------------------
# 5. Inspect Logistic Regression coefficients
# --------------------------------------------------

logistic_model = model.named_steps["logistic"]

coefficients = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": logistic_model.coef_[0]
})

coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)

coefficients = coefficients.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Logistic Regression feature coefficients:")
display(
    coefficients[
        ["feature", "coefficient", "absolute_coefficient"]
    ]
)


# --------------------------------------------------
# 6. Basic sanity check
# --------------------------------------------------

print("Model interpretation:")
for _, row in coefficients.iterrows():
    direction = "increases" if row["coefficient"] > 0 else "decreases"
    print(
        f"- {row['feature']}: "
        f"coefficient={row['coefficient']:.4f} "
        f"→ {direction} predicted decline probability"
    )

Error summary:


,count
error_type,
Correct,18823
False Positive,11750
False Negative,5449


Top 3 false positives:


,content_hash_id,model_score,is_declining,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
138931,content_8e1334d6356668e3,0.997033,False,119229.0,1.0,2.982337,410,<NA>
89721,content_425715547c6a3ea8,0.967646,False,68578.0,3.0,6.950334,371,<NA>
51406,content_91d8af19d84c05a6,0.876776,False,45936.0,26.0,6.051332,410,<NA>


Top 3 false negatives:


,content_hash_id,model_score,is_declining,imp_prev30,clicks_prev30,avg_position_prev30,content_age_days,days_since_last_update
34354,content_b2cb08ff59fcce78,0.000092,True,103526.0,647.0,3.148736,243,<NA>
44403,content_1fb7c75c360b4a7f,0.000862,True,100204.0,521.0,3.231278,56,<NA>
120797,content_e8b074fd4a082388,0.001279,True,54721.0,427.0,4.102246,260,<NA>


Logistic Regression feature coefficients:


,feature,coefficient,absolute_coefficient
1,clicks_prev30,-0.573542,0.573542
0,imp_prev30,0.239549,0.239549
3,content_age_days,0.222161,0.222161
2,avg_position_prev30,-0.121741,0.121741
4,days_since_last_update,-0.006134,0.006134


Model interpretation:
- clicks_prev30: coefficient=-0.5735 → decreases predicted decline probability
- imp_prev30: coefficient=0.2395 → increases predicted decline probability
- content_age_days: coefficient=0.2222 → increases predicted decline probability
- avg_position_prev30: coefficient=-0.1217 → decreases predicted decline probability
- days_since_last_update: coefficient=-0.0061 → decreases predicted decline probability


The error analysis shows that the model made **11,750 false positives** and **5,449 false negatives**, compared with **18,823 correct predictions**. This means the model produced more false-positive errors than false-negative errors on the held-out test set.

The false-positive examples show that the model can assign a high decline probability to highly visible pages that ultimately were not labeled as declining. For example, the three inspected false positives had `imp_prev30` values of 119,229, 68,578, and 45,936, while their model scores were 0.997, 0.968, and 0.877 respectively. This suggests that high previous-period impressions can contribute strongly to a high predicted decline probability even when the observed outcome does not meet the decline definition.

The false-negative examples show the opposite behavior. Three pages with very high previous-period impressions and substantial clicks were assigned extremely low decline probabilities even though they were labeled as declining. This indicates that the model can miss declines among pages that had strong previous-period performance.

The coefficient analysis shows that the model relies most on `clicks_prev30`, followed by `imp_prev30` and `content_age_days`, based on absolute standardized coefficient magnitude. `clicks_prev30` has the largest negative coefficient (-0.5735), while `imp_prev30` (0.2395) and `content_age_days` (0.2222) have positive coefficients. `avg_position_prev30` has a smaller negative coefficient (-0.1217), while `days_since_last_update` is nearly neutral (-0.0061).

Overall, the model appears to rely primarily on historical traffic and engagement signals rather than page freshness. The error analysis also shows that these signals are not sufficient to identify every observed decline correctly. These are model associations rather than causal effects, so the results should be treated as decision-support signals for prioritizing human review rather than proof that a page should be refreshed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.